# Real-data showcase

Four real-data case studies fit by BayesBreak, mirroring §6 of the manuscript:

1. **Well-log NMR** geology (Gaussian block, length 4050 NMR series).
2. **Coriell array-CGH** copy number (heteroscedastic multi-subject Gaussian).
3. **S&P 500 squared returns** volatility regimes (Gaussian on `log r_t^2`).
4. **CpG-atlas methylation** (Beta-response block with per-CpG precision).

Each loader falls back to a deterministic simulated analog when the network download is unavailable, so this notebook runs end-to-end on a fresh checkout without any datasets installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bayesbreak import BayesBreakGaussian, BayesBreakBetaObs
from bayesbreak.datasets import (
    load_welllog,
    load_cgh,
    load_spx,
    load_methylation,
)


## 1. Well-log NMR (Gaussian block)

In [ ]:
bundle = load_welllog()
print('source:', bundle.source, ' n:', bundle.y.size)

# Subsample 4050 -> ~500 for a fast demo.
y = bundle.y[::8]
X = np.arange(y.size).reshape(-1, 1)

est = BayesBreakGaussian(k_max=40, regression_curve='none').fit(X, y)
print(f'k_map={est.k_map_}, log p(y)={est.log_evidence_:.1f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].plot(X.ravel(), y, color='grey', lw=0.5)
axes[0].plot(X.ravel(), est.map_curve_, color='C0', lw=1.5, label='MAP')
for b in est.map_boundaries_[1:-1]:
    axes[0].axvline(b, color='C3', ls='--', lw=0.6, alpha=0.7)
axes[0].set_ylabel('NMR (standardised)')
axes[0].legend()

axes[1].fill_between(np.arange(1, y.size), 0, est.boundary_marginals_, color='C0', alpha=0.4)
axes[1].set_xlabel('index')
axes[1].set_ylabel(r'$P(b_i=1 | y)$')
plt.tight_layout(); plt.show()

## 2. Array-CGH (multi-subject pooled Gaussian)

In [ ]:
from bayesbreak import SharedBoundaryReplicatesSegmenter

cgh = load_cgh()
print('source:', cgh.source, ' shape:', cgh.y.shape)

y_cgh = cgh.y if cgh.y.ndim == 2 else cgh.y[:, None]
w_cgh = cgh.sample_weight
X_cgh = np.arange(y_cgh.shape[0]).reshape(-1, 1)

rep = SharedBoundaryReplicatesSegmenter(
    BayesBreakGaussian(k_max=15)
).fit(X_cgh, y_cgh, sample_weight=w_cgh)
print(f'pooled k_map={rep.k_map_}, log p(y)={rep.log_evidence_:.1f}')

## 3. S&P 500 volatility regimes (Gaussian on log squared returns)

In [ ]:
spx = load_spx()
print('source:', spx.source, ' n:', spx.y.size)

y_spx = spx.y[::4]              # stride-4 subsample for speed
X_spx = np.arange(y_spx.size).reshape(-1, 1)
est_spx = BayesBreakGaussian(k_max=50, regression_curve='none').fit(X_spx, y_spx)
print(f'k_map={est_spx.k_map_}, log p(y)={est_spx.log_evidence_:.1f}')

plt.figure(figsize=(8, 2.5))
plt.plot(X_spx.ravel(), y_spx, color='grey', lw=0.4)
for b in est_spx.map_boundaries_[1:-1]:
    plt.axvline(b, color='C3', ls='--', lw=0.5, alpha=0.6)
plt.title('SPX volatility-regime MAP boundaries'); plt.tight_layout(); plt.show()

## 4. Methylation (Beta-response block with per-CpG precision)

In [ ]:
meth = load_methylation()
print('source:', meth.source, ' n:', meth.y.size)

phi = meth.sample_weight if meth.sample_weight is not None else 50.0
est_meth = BayesBreakBetaObs(k_max=15, phi=phi, regression_curve='mix_k').fit(meth.X, meth.y)
print(f'k_map={est_meth.k_map_}, log p(y)={est_meth.log_evidence_:.1f}')

## What's next?

- See [Diagnostics walkthrough](04_diagnostics.ipynb) for the TV-bound, prior-sensitivity, and G-selection diagnostics.
- See [Baselines comparison](05_baselines.ipynb) for PELT / BS / WBS comparisons on the same data.
- See [Sliding-window for large n](06_sliding_window.ipynb) for handling sequences far longer than the exact DP can fit.